# Feeding External Data into a Pipeline with `InputsPassthrough`

The `TabularCSVFetcher` source node loads data from disk, but many real-world
workflows need to **inject data programmatically** — from a database query, an
API response, or a train/test split produced in-memory.

`InputsPassthrough` is a source node that does exactly that: you declare its
output ports, then pass the actual data at runtime via the `input_data`
argument of `train()`, `infer()`, and `evaluate()`.

This notebook covers:

1. Setting up an `InputsPassthrough` source with typed output ports
2. Building a pipeline that uses it
3. Passing data for **training**, **inference**, and **evaluation**
4. Making the target port inactive during inference (so you don't need labels)

## 1. Setup

In [1]:
import sys

import numpy as np
import pandas as pd

from tsut import NODE_REGISTRY
from tsut.core.common.logging import configure

configure(level="INFO", stream=sys.stdout, fmt="text")

NodeRegistry with 0 registered nodes: []
Successfully registered nodes from module: tsut.components.nodes.data_sources._register
Successfully registered nodes from module: tsut.components.nodes.metrics.classification._register
Successfully registered nodes from module: tsut.components.nodes.metrics.regression._register
Successfully registered nodes from module: tsut.components.nodes.models._register
Successfully registered nodes from module: tsut.components.nodes.sinks._register
Successfully registered nodes from module: tsut.components.nodes.transforms.encodings._register
Successfully registered nodes from module: tsut.components.nodes.transforms.feature_selection._register
Successfully registered nodes from module: tsut.components.nodes.transforms.filters._register
Successfully registered nodes from module: tsut.components.nodes.transforms.imputations._register
Successfully registered nodes from module: tsut.components.nodes.transforms.operations._register
Successfully registered nod

<Logger tsut (INFO)>

## 2. Create a synthetic dataset

We generate a simple regression problem: `y = 3*x1 + 2*x2 + noise`.

In [2]:
rng = np.random.default_rng(42)
n_samples = 200

x1 = rng.standard_normal(n_samples)
x2 = rng.standard_normal(n_samples)
y = 3.0 * x1 + 2.0 * x2 + rng.normal(0, 0.05, n_samples)

X_df = pd.DataFrame({"x1": x1, "x2": x2})
y_df = pd.DataFrame({"target": y})

print(f"X shape: {X_df.shape}")
print(f"y shape: {y_df.shape}")
X_df.head()

X shape: (200, 2)
y shape: (200, 1)


,x1,x2
0,0.304717,0.337575
1,-1.039984,1.407482
2,0.750451,0.090585
3,0.940565,0.643939
4,-1.951035,-2.050172


## 3. Configure the `InputsPassthrough` node

The key idea: we declare **output ports** on the source node that describe
the data we will provide at runtime. Each port specifies the array type,
data structure, category, and shape.

The **`y` port** is set to `mode=["training", "evaluation"]` — this tells
the runner that targets are not needed during inference, so the pipeline
can run without them.

In [3]:
from tsut.core.common.data.data import (
    DATA_CATEGORY_MAPPING,
    ArrayLikeEnum,
    DataCategoryEnum,
    DataStructureEnum,
    TabularDataContext,
)
from tsut.core.common.enums import NodeExecutionMode
from tsut.core.nodes.node import Port

passthrough_config_cls = NODE_REGISTRY.get_node_config_class("InputsPassthrough")

source_config = passthrough_config_cls(
    out_ports={
        "X": Port(
            arr_type=ArrayLikeEnum.PANDAS,
            data_structure=DataStructureEnum.TABULAR,
            data_category=DataCategoryEnum.NUMERICAL,
            data_shape="batch feature",
            desc="Feature matrix.",
        ),
        "y": Port(
            arr_type=ArrayLikeEnum.PANDAS,
            data_structure=DataStructureEnum.TABULAR,
            data_category=DataCategoryEnum.NUMERICAL,
            data_shape="batch 1",
            desc="Regression targets (not needed at inference time).",
            mode=[NodeExecutionMode.TRAINING, NodeExecutionMode.EVALUATION],
        ),
    },
)

print("Source ports:")
for name, port in source_config.out_ports.items():
    print(f"  {name}: mode={port.mode}")

Source ports:
  X: mode=[<NodeExecutionMode.ALL: 'all'>]
  y: mode=[<NodeExecutionMode.TRAINING: 'training'>, <NodeExecutionMode.EVALUATION: 'evaluation'>]


## 4. Build the pipeline

A minimal regression pipeline:

```
InputsPassthrough ──X──> StandardScaler ──> LinearRegression ──pred──> Sink
       └──y──────────────────────────────────> LinearRegression
       └──y──> R2Score  (evaluation only)
       └──────> LinearRegression ──pred──> R2Score
```

The `R2Score` metric node has its ports defaulted to evaluation mode,
so it is only walked during `evaluate()`.

In [4]:
from tsut.core.pipeline.pipeline import Edge, Pipeline, PipelineConfig

nodes = {
    "source": ("InputsPassthrough", source_config),
    "scaler": ("StandardScaler", NODE_REGISTRY.get_node_config_class("StandardScaler")()),
    "model": ("LinearRegression", NODE_REGISTRY.get_node_config_class("LinearRegression")()),
    "r2": ("R2Score", NODE_REGISTRY.get_node_config_class("R2Score")()),
    "sink": ("Sink", NODE_REGISTRY.get_node_config_class("Sink")()),
}

edges = [
    Edge(source="source", target="scaler", ports_map=[("X", "input")]),
    Edge(source="scaler", target="model", ports_map=[("output", "X")]),
    Edge(source="source", target="model", ports_map=[("y", "y")]),
    Edge(source="model", target="r2", ports_map=[("pred", "pred")]),
    Edge(source="source", target="r2", ports_map=[("y", "target")]),
    Edge(source="model", target="sink", ports_map=[("pred", "dump")]),
]

pipe = Pipeline(config=PipelineConfig(nodes=nodes, edges=edges, name="InputsPassthroughDemo"))

08:57:32 WARNING  [tsut.pipeline] Data structure mismatch (compatible but not identical)  source=model  target=sink  source_port=pred  target_port=dump


In [5]:
pipe.render()

## 5. Compile and create the runner

In [6]:
from tsut.core.pipeline.runners.smart_runner import SmartRunner

pipe.compile()
runner = SmartRunner(pipe)

08:57:32 INFO     [tsut.pipeline] Phase compilation start  pipeline_name=InputsPassthroughDemo  pipeline_phase=compilation  phase_status=start
08:57:32 WARNING  [tsut.pipeline] Data structure mismatch (compatible but not identical)  source=model  target=sink  source_port=pred  target_port=dump
08:57:32 INFO     [tsut.pipeline] Phase compilation end  pipeline_name=InputsPassthroughDemo  pipeline_phase=compilation  phase_status=end  params={'node_count': 5, 'edge_count': 6}


## 6. Prepare the `input_data` payload

The runner's `train()`, `infer()`, and `evaluate()` methods accept an
`input_data` dict with the following structure:

```python
input_data = {
    "<node_name>": {
        "<port_name>": (array, context),
        ...
    },
    ...
}
```

Each value is a tuple of:
- An **array** (`pd.DataFrame`, `np.ndarray`, or `torch.Tensor`)
- A **`TabularDataContext`** describing columns, dtypes, and categories

In [12]:
X_context = TabularDataContext(
    columns=list(X_df.columns),
    dtypes=[X_df[c].dtype for c in X_df.columns],
    categories=[DATA_CATEGORY_MAPPING[DataCategoryEnum.NUMERICAL]] * len(X_df.columns),
)

y_context = TabularDataContext(
    columns=list(y_df.columns),
    dtypes=[y_df[c].dtype for c in y_df.columns],
    categories=[DATA_CATEGORY_MAPPING[DataCategoryEnum.NUMERICAL]],
)

X_pair = (X_df, X_context)
y_pair = (y_df, y_context)

print(f"X context: columns={X_context.columns}, dtypes={X_context.dtypes}, categories={X_context.categories}")
print(f"y context: columns={y_context.columns}, dtypes={y_context.dtypes}, categories={y_context.categories}")

X context: columns=['x1', 'x2'], dtypes=[dtype('float64'), dtype('float64')], categories=[<class 'tsut.core.common.data.data.NumericalData'>, <class 'tsut.core.common.data.data.NumericalData'>]
y context: columns=['target'], dtypes=[dtype('float64')], categories=[<class 'tsut.core.common.data.data.NumericalData'>]


## 7. Training

During training, **all ports** marked as `training` or `all` are active.
We must supply both `X` and `y`.

In [8]:
runner.train(
    input_data={
        "source": {
            "X": X_pair,
            "y": y_pair,
        },
    },
)
print("Training complete.")

08:57:32 INFO     [tsut.runner.smart] Phase training start  pipeline_name=InputsPassthroughDemo  pipeline_version=0.1.0  pipeline_phase=training  phase_status=start


Training:   0%|          | 0/4 nodes [00:00<?, ?nodes/s]

08:57:32 INFO     [tsut.runner.smart] Node executed: source  pipeline_name=InputsPassthroughDemo  pipeline_version=0.1.0  node_name=source  pipeline_phase=training  duration_ms=0.4813330015167594  node_type=source
08:57:32 INFO     [tsut.runner.smart] Node executed: scaler  pipeline_name=InputsPassthroughDemo  pipeline_version=0.1.0  node_name=scaler  pipeline_phase=training  duration_ms=1.6938750632107258  node_type=transform
08:57:32 INFO     [tsut.runner.smart] Node executed: model  pipeline_name=InputsPassthroughDemo  pipeline_version=0.1.0  node_name=model  pipeline_phase=training  duration_ms=1.574208028614521  node_type=model
08:57:32 INFO     [tsut.runner.smart] Node executed: sink  pipeline_name=InputsPassthroughDemo  pipeline_version=0.1.0  node_name=sink  pipeline_phase=training  duration_ms=0.10454095900058746  node_type=sink
08:57:32 INFO     [tsut.runner.smart] Phase training end  pipeline_name=InputsPassthroughDemo  pipeline_version=0.1.0  pipeline_phase=training  phase_

## 8. Inference

During inference, the `y` port is **inactive** (its mode is
`[training, evaluation]`). The runner prunes every edge feeding into
inactive ports, so the pipeline runs without targets.

This means we only need to provide `X`.

In [9]:
predictions = runner.infer(
    input_data={
        "source": {
            "X": X_pair,
        },
    },
)

pred_df, pred_ctx = predictions["dump"]
print(f"Predictions shape: {pred_df.shape}")
pred_df.head()

08:57:32 INFO     [tsut.runner.smart] Phase inference start  pipeline_name=InputsPassthroughDemo  pipeline_version=0.1.0  pipeline_phase=inference  phase_status=start


Inference:   0%|          | 0/4 nodes [00:00<?, ?nodes/s]

08:57:32 INFO     [tsut.runner.smart] Node executed: source  pipeline_name=InputsPassthroughDemo  pipeline_version=0.1.0  node_name=source  pipeline_phase=inference  duration_ms=0.1450419658794999  node_type=source
08:57:32 INFO     [tsut.runner.smart] Node executed: scaler  pipeline_name=InputsPassthroughDemo  pipeline_version=0.1.0  node_name=scaler  pipeline_phase=inference  duration_ms=0.5479169776663184  node_type=transform
08:57:32 INFO     [tsut.runner.smart] Node executed: model  pipeline_name=InputsPassthroughDemo  pipeline_version=0.1.0  node_name=model  pipeline_phase=inference  duration_ms=0.41500001680105925  node_type=model
08:57:32 INFO     [tsut.runner.smart] Node executed: sink  pipeline_name=InputsPassthroughDemo  pipeline_version=0.1.0  node_name=sink  pipeline_phase=inference  duration_ms=0.12316706124693155  node_type=sink
08:57:32 INFO     [tsut.runner.smart] Phase inference end  pipeline_name=InputsPassthroughDemo  pipeline_version=0.1.0  pipeline_phase=inference

,target
0,1.585524
1,-0.303666
2,2.427222
3,4.104636
4,-9.955119


In [10]:
residuals = y_df["target"].values - pred_df.values.flatten()
print(f"Mean absolute error: {np.mean(np.abs(residuals)):.4f}")
print(f"Max absolute error:  {np.max(np.abs(residuals)):.4f}")

Mean absolute error: 0.0402
Max absolute error:  0.1470


## 9. Evaluation

During evaluation, the `y` port is active again (mode includes `evaluation`).
The `R2Score` metric node is also walked — it receives the model's
predictions and the ground-truth targets.

`evaluate()` returns a dict mapping metric node names to their computed
outputs.

In [11]:
metrics = runner.evaluate(
    input_data={
        "source": {
            "X": X_pair,
            "y": y_pair,
        },
    },
)

for metric_name, (score_df, score_ctx) in metrics.items():
    score = float(score_df.to_numpy().flatten()[0])
    print(f"{metric_name}: {score:.6f}")

08:57:32 INFO     [tsut.runner.smart] Phase evaluation start  pipeline_name=InputsPassthroughDemo  pipeline_version=0.1.0  pipeline_phase=evaluation  phase_status=start


Evaluation:   0%|          | 0/4 nodes [00:00<?, ?nodes/s]

08:57:32 INFO     [tsut.runner.smart] Node executed: source  pipeline_name=InputsPassthroughDemo  pipeline_version=0.1.0  node_name=source  pipeline_phase=evaluation  duration_ms=0.19216700457036495  node_type=source
08:57:32 INFO     [tsut.runner.smart] Node executed: scaler  pipeline_name=InputsPassthroughDemo  pipeline_version=0.1.0  node_name=scaler  pipeline_phase=evaluation  duration_ms=0.4937499761581421  node_type=transform
08:57:32 INFO     [tsut.runner.smart] Node executed: model  pipeline_name=InputsPassthroughDemo  pipeline_version=0.1.0  node_name=model  pipeline_phase=evaluation  duration_ms=0.5467080045491457  node_type=model
08:57:32 INFO     [tsut.runner.smart] Node executed: r2  pipeline_name=InputsPassthroughDemo  pipeline_version=0.1.0  node_name=r2  pipeline_phase=evaluation  duration_ms=2.9832080472260714  node_type=metric
08:57:32 INFO     [tsut.runner.smart] Phase evaluation end  pipeline_name=InputsPassthroughDemo  pipeline_version=0.1.0  pipeline_phase=evaluat

## Summary

| Phase       | Ports active on source | Data to provide | Returns           |
|-------------|------------------------|-----------------|-------------------|
| `train()`   | `X`, `y`               | Features + targets | Nothing           |
| `infer()`   | `X`                    | Features only      | Sink outputs      |
| `evaluate()`| `X`, `y`               | Features + targets | Metric scores     |

The key takeaway: by setting `mode=["training", "evaluation"]` on the
target port, the pipeline **automatically skips** target-dependent nodes
during inference — no `if/else` logic needed in your code.